# ⊙ Masterclass RAG · Notebook Práctico

Ejecuta cada celda en orden y lee los comentarios del código.
Al final de cada fase hay preguntas de reflexión, no hay respuesta correcta única, son para que pienses.

**Tiempo estimado: 25–30 minutos**

| Fase | Técnica | Lo que verás |
|------|---------|--------------|
| 01 | Búsqueda Híbrida | Por qué Dense solo falla con nombres propios |
| 02 | Reranking | Por qué el orden importa tanto como encontrar el doc |
| 03 | CRAG | Por qué un doc correcto puede ser aún insuficiente |

---
## 0 · Instalación

Instalamos dos librerías externas que no vienen con Python por defecto:

- **`sentence-transformers`** → carga modelos de lenguaje preentrenados que convierten texto en vectores numéricos (embeddings). La usamos para Dense Retrieval y para el Cross-Encoder del Reranking.
- **`rank_bm25`** → implementación del algoritmo BM25, el estándar de búsqueda por palabras clave. No usa redes neuronales: solo cuenta y pondera coincidencias de texto.

Ejecuta esta celda una vez y espera ~2 minutos.

In [1]:
!pip install -q sentence-transformers rank_bm25

In [2]:
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

print("✓ Dependencias listas")

✓ Dependencias listas


---
## 1 · La base de conocimiento

Estos son los documentos que nuestro sistema RAG tiene disponibles.
Fíjate que para cada pregunta de prueba existe **un doc correcto** y **un doc trampa**
que suena relacionado pero daría una respuesta incompleta o incorrecta.

In [3]:
# Cada documento simula un "chunk" de una base de conocimiento real
# En producción serían fragmentos de PDFs, wikis, manuales, etc.
DOCS = [
    {"id": "d1", "titulo": "Lynel Escarlata: anatomía y patrones de ataque",
     "texto": "El Lynel Escarlata es uno de los enemigos más fuertes de Hyrule. Tiene tres fases: carga, combo de espada y llamarada. Su punto débil está en la cabeza durante el salto."},

    {"id": "d2", "titulo": "Cómo derrotar al Lynel Escarlata: guía táctica",  # CORRECTO para Q1
     "texto": "Para vencer al Lynel Escarlata: ejecuta una esquiva perfecta justo antes de su carga para activar cámara lenta, dispara flechas a la cabeza y luego monta su lomo. Nunca bloquees su combo de espada, esquiva lateralmente siempre."},

    {"id": "d3", "titulo": "Técnicas avanzadas de combate cuerpo a cuerpo",   # TRAMPA para Q1
     "texto": "Las técnicas avanzadas incluyen el parry perfecto con escudo, la esquiva lateral cronometrada y el contraataque inmediato. Son efectivas contra jefes de alta dificultad y minimizan el daño recibido."},

    {"id": "d4", "titulo": "Templo del Agua de Zora: acceso y ubicación",     # CORRECTO para Q2
     "texto": "El Templo del Agua de Zora se accede desde el Dominio Zora, al noreste de Hyrule. Requiere la Escama de Zora para entrar. Coordenadas: X=2365, Y=0412. El jefe interior es Morpha."},

    {"id": "d5", "titulo": "Guía de reinos y zonas acuáticas de Hyrule",      # TRAMPA para Q2
     "texto": "Las zonas acuáticas de Hyrule incluyen el Lago Hylia, el Dominio Zora y las Cataratas de Zora. Para explorarlas necesitas equipamiento de natación o la habilidad Zora."},

    {"id": "d6", "titulo": "Elixir Ígnifrugal: receta y propiedades",         # CORRECTO para Q3
     "texto": "El Elixir Ígnifrugal se fabrica con Polvo de Salamandra de Fuego x3 más cuerpo de insecto. Otorga resistencia al fuego nivel 2 durante 8 minutos. No requiere olla, se prepara en mortero de campo."},

    {"id": "d7", "titulo": "Manual de alquimia y recetas de campo de Hyrule", # TRAMPA para Q3
     "texto": "Las recetas de campo permiten fabricar elixires sin fuego usando ingredientes naturales. Combina insectos con lagartijas o materiales de monstruo para crear efectos de resistencia elemental."},

    {"id": "d8", "titulo": "Armadura Ígnifruga del Guerrero Goron",
     "texto": "La Armadura Ígnifruga Goron otorga inmunidad total al fuego. Se compra en la Herrería de Goron City por 2400 rupias y no puede fabricarse."},
]

# Las 3 preguntas con las que vamos a evaluar el sistema
PREGUNTAS = [
    {"query": "¿Cómo derrotar al Lynel Escarlata?",         "correcto": "d2", "trampa": "d3"},
    {"query": "¿Cómo acceder al Templo del Agua de Zora?",  "correcto": "d4", "trampa": "d5"},
    {"query": "¿Cómo fabricar el Elixir Ígnifrugal?",       "correcto": "d6", "trampa": "d7"},
]

print(f"Base de conocimiento: {len(DOCS)} documentos")
print(f"Preguntas de prueba : {len(PREGUNTAS)}")

Base de conocimiento: 8 documentos
Preguntas de prueba : 3


---
## 2 · El problema del RAG básico

### ¿Qué es un embedding?

Un **embedding** es la traducción de un texto a una lista de números (un vector).
El modelo aprende a colocar textos con significados similares cerca en ese espacio numérico:

```
"perro"   →  [0.21, -0.54, 0.88, ...]   ─┐
"canino"  →  [0.19, -0.51, 0.85, ...]   ─┴─ cerca (semánticamente similares)

"receta"  →  [-0.73, 0.12, -0.44, ...]  ─── lejos de "perro"
```

La **similitud coseno** mide el ángulo entre dos vectores: si el ángulo es pequeño (vectores apuntando en la misma dirección), los textos son similares. Con vectores normalizados, esto se reduce a un simple producto punto.

### El punto ciego

Este enfoque funciona bien para lenguaje natural, pero falla con términos específicos:
el modelo no sabe que **"Lynel Escarlata"** es un nombre propio único,
lo trata como "una bestia" genérica y puede traer documentos sobre combate en general
cuando lo que necesitas es información específica de ese enemigo concreto.

Observa qué pasa:

In [4]:
# Cargamos el modelo de embeddings
# Este modelo convierte cualquier texto en un vector de 384 números
# que representan su "posición" en un espacio semántico
print("Cargando modelo de embeddings... (~1 min)")
modelo = SentenceTransformer("all-MiniLM-L6-v2")

# Convertimos todos los documentos a vectores de una vez (batch)
# Concatenamos título + texto para que el vector capture ambos
textos = [f"{d['titulo']}. {d['texto']}" for d in DOCS]
vectores_docs = modelo.encode(textos, normalize_embeddings=True)
# normalize_embeddings=True → cada vector tiene longitud 1
# esto hace que el producto punto sea equivalente a la similitud coseno

print(f"✓ {len(DOCS)} documentos convertidos a vectores de {vectores_docs.shape[1]} dimensiones")

Cargando modelo de embeddings... (~1 min)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ 8 documentos convertidos a vectores de 384 dimensiones


In [5]:
def buscar_dense(query, top_k=4):
    # Convertimos la pregunta al mismo espacio vectorial que los docs
    vector_query = modelo.encode([query], normalize_embeddings=True)

    # Similitud coseno: qué tan "parecido" es el vector de la query a cada doc
    # Con vectores normalizados, el producto punto = similitud coseno
    scores = (vector_query @ vectores_docs.T)[0]

    # Ordenamos de mayor a menor y tomamos los top_k
    indices = np.argsort(scores)[::-1][:top_k]
    return [(DOCS[i], float(scores[i])) for i in indices]


# ── Probamos con la primera pregunta ──────────────────────────
query = PREGUNTAS[0]["query"]
print(f"Query: '{query}'")
print("-" * 65)

for i, (doc, score) in enumerate(buscar_dense(query), 1):
    marca = ""
    if doc["id"] == PREGUNTAS[0]["correcto"]: marca = "  ← CORRECTO"
    if doc["id"] == PREGUNTAS[0]["trampa"]:   marca = "  ← TRAMPA ⚠"
    print(f"  #{i}  [{score:.3f}]  {doc['titulo']}{marca}")

Query: '¿Cómo derrotar al Lynel Escarlata?'
-----------------------------------------------------------------
  #1  [0.743]  Lynel Escarlata: anatomía y patrones de ataque
  #2  [0.740]  Cómo derrotar al Lynel Escarlata: guía táctica  ← CORRECTO
  #3  [0.397]  Técnicas avanzadas de combate cuerpo a cuerpo  ← TRAMPA ⚠
  #4  [0.370]  Templo del Agua de Zora: acceso y ubicación


### ⊙ Reflexiona

> **1.** El modelo probablemente colocó "Técnicas avanzadas de combate cuerpo a cuerpo" en una posición alta.
> ¿Por qué crees que eso ocurre si ese documento no habla específicamente del Lynel Escarlata?

> **2.** "Lynel Escarlata", "Templo del Agua de Zora" y "Elixir Ígnifrugal" son nombres propios.
> ¿Qué dificultad especial tienen los nombres propios para un modelo semántico?

> **3.** ¿En qué tipo de base de conocimiento de empresa crees que este problema sería más grave:
> en una wiki de RRHH o en documentación técnica con códigos de producto?

---
## 3 · Fase 01: Búsqueda Híbrida

La solución es combinar dos motores complementarios:

- **Dense** → entiende el significado. Bueno para lenguaje natural y paráfrasis.
- **BM25** → cuenta coincidencias exactas de palabras. Bueno para términos técnicos y nombres propios.

Luego los fusionamos con **RRF**: un documento que aparece bien posicionado en *ambas* listas gana.

In [6]:
# BM25 trabaja sobre tokens (palabras), no sobre vectores
# Por eso tokenizamos simplemente separando por espacios
corpus_tokenizado = [f"{d['titulo']} {d['texto']}".lower().split() for d in DOCS]
bm25 = BM25Okapi(corpus_tokenizado)

def buscar_bm25(query, top_k=4):
    # BM25 puntúa cada doc según cuántas palabras de la query contiene
    # y con qué frecuencia aparecen respecto al tamaño del documento
    scores = bm25.get_scores(query.lower().split())
    indices = np.argsort(scores)[::-1][:top_k]
    return [(DOCS[i], float(scores[i])) for i in indices]


# ── Comparamos Dense vs BM25 en la misma query ────────────────
query = PREGUNTAS[0]["query"]
res_dense = buscar_dense(query, top_k=4)
res_bm25  = buscar_bm25(query, top_k=4)

print(f"Query: '{query}'\n")
print(f"{'#':<3}  {'DENSE (semántico)':<40}  BM25 (exacto)")
print("-" * 80)
for i in range(4):
    d = res_dense[i][0]["titulo"][:37]
    b = res_bm25[i][0]["titulo"][:37]
    print(f"#{i+1:<2}  {d:<40}  {b}")

print("\n⊙ Nota: BM25 detecta 'Lynel Escarlata' como término exacto → sube el doc correcto")

Query: '¿Cómo derrotar al Lynel Escarlata?'

#    DENSE (semántico)                         BM25 (exacto)
--------------------------------------------------------------------------------
#1   Lynel Escarlata: anatomía y patrones      Cómo derrotar al Lynel Escarlata: guí
#2   Cómo derrotar al Lynel Escarlata: guí     Lynel Escarlata: anatomía y patrones 
#3   Técnicas avanzadas de combate cuerpo      Armadura Ígnifruga del Guerrero Goron
#4   Templo del Agua de Zora: acceso y ubi     Manual de alquimia y recetas de campo

⊙ Nota: BM25 detecta 'Lynel Escarlata' como término exacto → sube el doc correcto


In [7]:
def fusionar_rrf(*rankings, k=60):
    # RRF asigna a cada doc un score basado en su posición en cada lista
    # Score = Σ 1 / (k + posición)
    # k=60 es el valor estándar: suaviza la diferencia entre posición #1 y #2
    scores = {}
    for ranking in rankings:
        for posicion, (doc, _) in enumerate(ranking, 1):
            id_doc = doc["id"]
            # Un doc en posición 1 aporta: 1/(60+1) = 0.016
            # Un doc en posición 5 aporta: 1/(60+5) = 0.015
            # La diferencia es pequeña → evita que un solo motor domine
            scores[id_doc] = scores.get(id_doc, 0) + 1.0 / (k + posicion)

    ordenados = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    doc_por_id = {d["id"]: d for d in DOCS}
    return [(doc_por_id[doc_id], score) for doc_id, score in ordenados]


res_hibrido = fusionar_rrf(res_dense, res_bm25)

print(f"Resultado HÍBRIDO (Dense + BM25 + RRF)\n")
print(f"{'#':<3}  {'Documento':<45}  Score RRF")
print("-" * 65)
for i, (doc, score) in enumerate(res_hibrido[:4], 1):
    marca = ""
    if doc["id"] == PREGUNTAS[0]["correcto"]: marca = "  ← CORRECTO"
    if doc["id"] == PREGUNTAS[0]["trampa"]:   marca = "  ← TRAMPA"
    print(f"#{i:<2}  {doc['titulo']:<45}  {score:.5f}{marca}")

Resultado HÍBRIDO (Dense + BM25 + RRF)

#    Documento                                      Score RRF
-----------------------------------------------------------------
#1   Lynel Escarlata: anatomía y patrones de ataque  0.03252
#2   Cómo derrotar al Lynel Escarlata: guía táctica  0.03252  ← CORRECTO
#3   Técnicas avanzadas de combate cuerpo a cuerpo  0.01587  ← TRAMPA
#4   Armadura Ígnifruga del Guerrero Goron          0.01587


### ⊙ Experimenta

Cambia el valor de `k` en la celda de abajo y vuelve a ejecutar.
¿Cambia el orden? ¿Se vuelve más o menos estable el ranking?

In [8]:
# ══ ZONA DE EXPERIMENTO ══════════════════════════════════════
K_RRF = 60   # prueba con 5 (posición importa mucho) o con 200 (posición importa poco)
# ═════════════════════════════════════════════════════════════

resultado = fusionar_rrf(res_dense, res_bm25, k=K_RRF)

print(f"Ranking con k={K_RRF}:")
for i, (doc, score) in enumerate(resultado[:4], 1):
    marca = " ✓" if doc["id"] == PREGUNTAS[0]["correcto"] else " ✗" if doc["id"] == PREGUNTAS[0]["trampa"] else ""
    print(f"  #{i}  {doc['titulo']}{marca}")

Ranking con k=60:
  #1  Lynel Escarlata: anatomía y patrones de ataque
  #2  Cómo derrotar al Lynel Escarlata: guía táctica ✓
  #3  Técnicas avanzadas de combate cuerpo a cuerpo ✗
  #4  Armadura Ígnifruga del Guerrero Goron


In [9]:
# ── Métrica: Hit Rate @3 ──────────────────────────────────────
# Hit Rate = fracción de preguntas donde el doc correcto aparece en los top-3
# Si es 0% el sistema ni siquiera encuentra el documento relevante

def hit_rate(preguntas, buscar_fn, k=3):
    aciertos = sum(
        p["correcto"] in [doc["id"] for doc, _ in buscar_fn(p["query"])[:k]]
        for p in preguntas
    )
    return aciertos / len(preguntas)

def hibrida(query, k=5):
    return fusionar_rrf(buscar_dense(query, k), buscar_bm25(query, k))[:k]

hr_naive    = hit_rate(PREGUNTAS, lambda q: buscar_dense(q, 3))
hr_hibrida  = hit_rate(PREGUNTAS, lambda q: hibrida(q, 3))

print(f"Hit Rate @3  —  Dense solo: {hr_naive:.0%}   Híbrida: {hr_hibrida:.0%}")
print(f"Mejora: +{(hr_hibrida - hr_naive)*100:.0f} puntos porcentuales")

Hit Rate @3  —  Dense solo: 100%   Híbrida: 100%
Mejora: +0 puntos porcentuales


### ⊙ Reflexiona

> **1.** RRF beneficia a los documentos que aparecen bien en **ambas** listas.
> ¿Por qué ese criterio es más robusto que confiar en un solo motor?

> **2.** ¿Qué pasó cuando cambiaste el valor de k?
> ¿Cuándo te conviene un k pequeño y cuándo uno grande?

> **3.** En tu empresa, ¿qué tipo de términos crees que BM25 rescataría mejor que Dense?
> Piensa en siglas, nombres de clientes, códigos internos...

---
## 4 · Fase 02: Reranking

Encontrar el documento correcto en top-3 no es suficiente si está en posición #3.
Los LLMs sufren el efecto **"Lost in the Middle"**: prestan más atención al principio
y al final del contexto. Un doc en posición #3 puede ser ignorado.

El **Cross-Encoder** lee la pregunta y el documento *juntos* en una sola pasada,
lo que le permite entender la relevancia con mucha más precisión que el Dense Retrieval.

In [10]:
print("Cargando Cross-Encoder... (~1 min)")
# Este modelo fue entrenado específicamente para dar un score de relevancia
# a pares (pregunta, documento). Es más lento pero mucho más preciso.
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("✓ Cross-Encoder listo")

Cargando Cross-Encoder... (~1 min)


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✓ Cross-Encoder listo


In [11]:
def reranking(query, candidatos, top_n=3):
    # Construimos pares (pregunta, texto_doc) para cada candidato
    pares = [(query, f"{doc['titulo']}. {doc['texto']}") for doc, _ in candidatos]

    # El Cross-Encoder evalúa todos los pares y devuelve un score por par
    # A diferencia del Dense, aquí la pregunta y el doc se procesan JUNTOS
    scores = cross_encoder.predict(pares)

    # Emparejamos cada doc con su nuevo score y ordenamos
    con_score = [(candidatos[i][0], float(scores[i])) for i in range(len(candidatos))]
    return sorted(con_score, key=lambda x: x[1], reverse=True)[:top_n]


# ── Comparamos antes y después del reranking ──────────────────
query = PREGUNTAS[0]["query"]
candidatos = hibrida(query, k=5)
rerankeados = reranking(query, candidatos)

print("ANTES del reranking (Híbrida top-4):")
for i, (doc, _) in enumerate(candidatos[:4], 1):
    marca = "  ← CORRECTO" if doc["id"] == PREGUNTAS[0]["correcto"] else ""
    marca = "  ← TRAMPA" if doc["id"] == PREGUNTAS[0]["trampa"] else marca
    print(f"  #{i}  {doc['titulo']}{marca}")

print("\nDESPUÉS del reranking (Cross-Encoder top-3):")
for i, (doc, score) in enumerate(rerankeados, 1):
    marca = "  ← CORRECTO" if doc["id"] == PREGUNTAS[0]["correcto"] else ""
    marca = "  ← TRAMPA" if doc["id"] == PREGUNTAS[0]["trampa"] else marca
    print(f"  #{i}  [{score:+.2f}]  {doc['titulo']}{marca}")

ANTES del reranking (Híbrida top-4):
  #1  Lynel Escarlata: anatomía y patrones de ataque
  #2  Cómo derrotar al Lynel Escarlata: guía táctica  ← CORRECTO
  #3  Técnicas avanzadas de combate cuerpo a cuerpo  ← TRAMPA
  #4  Armadura Ígnifruga del Guerrero Goron

DESPUÉS del reranking (Cross-Encoder top-3):
  #1  [+8.27]  Cómo derrotar al Lynel Escarlata: guía táctica  ← CORRECTO
  #2  [+5.17]  Lynel Escarlata: anatomía y patrones de ataque
  #3  [-10.43]  Templo del Agua de Zora: acceso y ubicación


### ⊙ Experimenta

Cambia cuántos candidatos le pasas al Cross-Encoder.
¿Hay diferencia entre pasarle 3 vs 7?

In [12]:
# ══ ZONA DE EXPERIMENTO ══════════════════════════════════════
CANDIDATOS_PARA_RERANKER = 5   # prueba con 3 o con 7
DOCS_AL_LLM              = 3   # cuántos pasan al LLM después del reranking
# ═════════════════════════════════════════════════════════════

for p in PREGUNTAS:
    cands = hibrida(p["query"], k=CANDIDATOS_PARA_RERANKER)
    top   = reranking(p["query"], cands, top_n=DOCS_AL_LLM)
    top1  = top[0][0]
    marca = "✓ CORRECTO" if top1["id"] == p["correcto"] else "✗ INCORRECTO"
    print(f"  {marca}  |  {p['query'][:45]:<45}  →  {top1['titulo'][:40]}")

  ✓ CORRECTO  |  ¿Cómo derrotar al Lynel Escarlata?             →  Cómo derrotar al Lynel Escarlata: guía t
  ✓ CORRECTO  |  ¿Cómo acceder al Templo del Agua de Zora?      →  Templo del Agua de Zora: acceso y ubicac
  ✓ CORRECTO  |  ¿Cómo fabricar el Elixir Ígnifrugal?           →  Elixir Ígnifrugal: receta y propiedades


In [13]:
# ── Métrica: MRR (Mean Reciprocal Rank) ──────────────────────
# MRR mide en qué posición exacta está el doc correcto
# Posición 1 → 1.0 (perfecto), posición 2 → 0.5, posición 3 → 0.33

def pipeline_hibrida_rerank(query, k=5):
    return reranking(query, hibrida(query, k), top_n=k)

def mrr(preguntas, buscar_fn, k=5):
    total = 0.0
    for p in preguntas:
        for pos, (doc, _) in enumerate(buscar_fn(p["query"])[:k], 1):
            if doc["id"] == p["correcto"]:
                total += 1.0 / pos
                break
    return total / len(preguntas)

mrr_naive    = mrr(PREGUNTAS, lambda q: buscar_dense(q, 5))
mrr_hibrida  = mrr(PREGUNTAS, lambda q: hibrida(q, 5))
mrr_reranked = mrr(PREGUNTAS, pipeline_hibrida_rerank)

print(f"MRR  —  Dense: {mrr_naive:.2f}   Híbrida: {mrr_hibrida:.2f}   + Reranking: {mrr_reranked:.2f}")
print(f"Mejora total vs básico: +{(mrr_reranked - mrr_naive)*100:.1f} puntos")

MRR  —  Dense: 0.83   Híbrida: 0.83   + Reranking: 1.00
Mejora total vs básico: +16.7 puntos


### ⊙ Reflexiona

> **1.** El Cross-Encoder es más lento porque procesa pregunta + documento juntos.
> El Dense Retrieval los procesa por separado. ¿Por qué esa diferencia hace al Cross-Encoder más preciso?

> **2.** Si tu sistema recibe 1000 preguntas por hora, ¿aplicarías reranking a todas?
> ¿Para qué tipo de preguntas sería prioritario?

> **3.** ¿Qué diferencia hay entre un MRR de 0.5 y un MRR de 1.0 en términos prácticos
> para el usuario final que espera una respuesta?

---
## 5 · Fase 03: CRAG (Corrective RAG)

Incluso con el mejor documento en #1, puede ocurrir que ese documento esté
**desactualizado**, sea **parcialmente incorrecto** o simplemente **no cubra la pregunta**.

En RAG básico, el LLM respondería igual, inventando la parte que falta: una **alucinación**.

CRAG agrega un **LLM Juez** que evalúa el documento antes de pasárselo al LLM generador:

| Score | Veredicto | Acción |
|-------|-----------|--------|
| ≥ 0.80 | CORRECTO | Pasa directo al LLM |
| 0.40 – 0.79 | AMBIGUO | Documento + búsqueda web |
| < 0.40 | INCORRECTO | Solo búsqueda web |

In [14]:
# En producción el LLM Juez es una llamada real a Claude Haiku o GPT-4-mini
# Aquí usamos scores precalculados para no necesitar API key
# El comportamiento es idéntico: score = qué tan bien responde el doc la query
SCORES_JUEZ = {
    ("¿Cómo derrotar al Lynel Escarlata?",         "d1"): 0.55,  # anatomía → falta táctica completa
    ("¿Cómo derrotar al Lynel Escarlata?",         "d2"): 0.94,  # guía táctica completa ✓
    ("¿Cómo derrotar al Lynel Escarlata?",         "d3"): 0.48,  # combate genérico → impreciso
    ("¿Cómo acceder al Templo del Agua de Zora?",  "d4"): 0.67,  # ubicación ok, requisitos posiblemente desactualizados
    ("¿Cómo acceder al Templo del Agua de Zora?",  "d5"): 0.31,  # genérico → sin acceso específico
    ("¿Cómo fabricar el Elixir Ígnifrugal?",       "d6"): 0.91,  # receta exacta ✓
    ("¿Cómo fabricar el Elixir Ígnifrugal?",       "d7"): 0.43,  # genérico → sin ingredientes exactos
}

def evaluar_crag(query, documento, umbral_correcto=0.80, umbral_ambiguo=0.40):
    score = SCORES_JUEZ.get((query, documento["id"]), 0.5)

    if score >= umbral_correcto:
        return score, "CORRECTO",    "→ pasa directo al LLM"
    elif score >= umbral_ambiguo:
        return score, "AMBIGUO",     "→ doc interno + búsqueda web"
    else:
        return score, "INCORRECTO",  "→ solo búsqueda web"


# ── Evaluamos las 3 preguntas ─────────────────────────────────
print("CRAG evalúa el documento top-1 del pipeline:\n")

for p in PREGUNTAS:
    cands = hibrida(p["query"], k=5)
    top1, _ = reranking(p["query"], cands, top_n=1)[0]
    score, veredicto, accion = evaluar_crag(p["query"], top1)
    icono = {"CORRECTO": "✓", "AMBIGUO": "◎", "INCORRECTO": "✗"}[veredicto]
    print(f"  {icono} [{score:.2f}]  {p['query']}")
    print(f"         Doc top-1: {top1['titulo']}")
    print(f"         {accion}\n")

CRAG evalúa el documento top-1 del pipeline:

  ✓ [0.94]  ¿Cómo derrotar al Lynel Escarlata?
         Doc top-1: Cómo derrotar al Lynel Escarlata: guía táctica
         → pasa directo al LLM

  ◎ [0.67]  ¿Cómo acceder al Templo del Agua de Zora?
         Doc top-1: Templo del Agua de Zora: acceso y ubicación
         → doc interno + búsqueda web

  ✓ [0.91]  ¿Cómo fabricar el Elixir Ígnifrugal?
         Doc top-1: Elixir Ígnifrugal: receta y propiedades
         → pasa directo al LLM



### ⊙ Experimenta

Sube el umbral de CORRECTO a `0.98`. ¿Qué pasa con las 3 preguntas?
¿Qué implicación tiene esto en coste y velocidad?

In [15]:
# ══ ZONA DE EXPERIMENTO ══════════════════════════════════════
UMBRAL_CORRECTO = 0.80   # prueba con 0.98 — ¿cuántas activan búsqueda web?
UMBRAL_AMBIGUO  = 0.40   # prueba con 0.60 — ¿cambia algún veredicto?
# ═════════════════════════════════════════════════════════════

sin_web = 0
con_web = 0

for p in PREGUNTAS:
    cands = hibrida(p["query"], k=5)
    top1, _ = reranking(p["query"], cands, top_n=1)[0]
    score, veredicto, accion = evaluar_crag(p["query"], top1, UMBRAL_CORRECTO, UMBRAL_AMBIGUO)
    icono = {"CORRECTO": "✓", "AMBIGUO": "◎", "INCORRECTO": "✗"}[veredicto]

    if veredicto == "CORRECTO": sin_web += 1
    else: con_web += 1

    print(f"  {icono} [{score:.2f}]  {p['query'][:45]:<45}  {accion}")

print(f"\n  Sin búsqueda web: {sin_web}/3   Con búsqueda web: {con_web}/3")

  ✓ [0.94]  ¿Cómo derrotar al Lynel Escarlata?             → pasa directo al LLM
  ◎ [0.67]  ¿Cómo acceder al Templo del Agua de Zora?      → doc interno + búsqueda web
  ✓ [0.91]  ¿Cómo fabricar el Elixir Ígnifrugal?           → pasa directo al LLM

  Sin búsqueda web: 2/3   Con búsqueda web: 1/3


### ⊙ Reflexiona

> **1.** Un umbral muy alto (0.98) activa búsqueda web casi siempre.
> Un umbral muy bajo (0.20) casi nunca la activa.
> ¿Cuál sería el criterio correcto para elegir el umbral en un sistema real?

> **2.** ¿En qué industria o aplicación crees que el coste de una alucinación justifica
> siempre activar CRAG completo? ¿En cuál no lo justifica?

> **3.** El LLM Juez consume tokens (= dinero) en cada evaluación.
> ¿Cómo optimizarías esto si tienes 10.000 preguntas al día?

---
## 6 · Comparativa final

Medimos el impacto acumulado de las tres fases sobre las mismas 3 preguntas.

**Cómo leer la tabla:**
- **Hit Rate @3** → ¿El doc correcto aparece en algún lugar del top-3? El pipeline básico puede tenerlo pero enterrado; la Búsqueda Híbrida se asegura de que siempre esté dentro del top-3.
- **MRR** → ¿En qué posición exacta está? Un MRR de 1.0 significa que el doc correcto es siempre el #1. El Reranking es el responsable de llevarlo a la primera posición.
- **Faithfulness** → ¿El doc que finalmente llega al LLM es el correcto? CRAG cierra el pipeline descartando o complementando cualquier doc que siga siendo insuficiente.

Un sistema con Hit Rate alto pero MRR bajo tiene los documentos pero desordenados.
Un sistema con MRR alto pero Faithfulness bajo encontró y ordenó bien, pero el LLM aún puede alucinar.

In [16]:
def pipeline_completo(query, k=5):
    # Fase 01: Híbrida
    candidatos = fusionar_rrf(buscar_dense(query, k), buscar_bm25(query, k))[:k]
    # Fase 02: Reranking
    return reranking(query, candidatos, top_n=k)

# ── Hit Rate @3 ───────────────────────────────────────────────
hr_naive    = hit_rate(PREGUNTAS, lambda q: buscar_dense(q, 3))
hr_avanzado = hit_rate(PREGUNTAS, lambda q: pipeline_completo(q, 5)[:3])

# ── MRR ──────────────────────────────────────────────────────
mrr_naive    = mrr(PREGUNTAS, lambda q: buscar_dense(q, 5))
mrr_avanzado = mrr(PREGUNTAS, pipeline_completo)

# ── Faithfulness (proxy) ──────────────────────────────────────
# ¿El doc que llega al LLM es el correcto?
faith = sum(
    pipeline_completo(p["query"])[0][0]["id"] == p["correcto"]
    for p in PREGUNTAS
) / len(PREGUNTAS)

print("╔═════════════════════════════════════════════════════╗")
print("║           COMPARATIVA FINAL · 3 PREGUNTAS          ║")
print("╠══════════════════╦═════════╦═══════════╦═══════════╣")
print("║  Métrica         ║  Básico ║ Pipeline  ║  Mejora   ║")
print("╠══════════════════╬═════════╬═══════════╬═══════════╣")
print(f"║  Hit Rate @3     ║  {hr_naive:>5.0%}  ║   {hr_avanzado:>5.0%}   ║  +{(hr_avanzado-hr_naive)*100:.0f} pp      ║")
print(f"║  MRR             ║  {mrr_naive:>5.2f}  ║   {mrr_avanzado:>5.2f}   ║  +{(mrr_avanzado-mrr_naive)*100:.1f} pp     ║")
print(f"║  Faithfulness    ║    N/A  ║   {faith:>5.0%}   ║   —         ║")
print("╚══════════════════╩═════════╩═══════════╩═══════════╝")
print()
print("  Hit Rate: Fase 01 (Búsqueda Híbrida)  asegura que el doc esté en el top-3")
print("  MRR:      Fase 02 (Reranking)         lo sube a la posición #1")
print("  Faith.:   Fase 03 (CRAG)              verifica que lo que llega al LLM sea correcto")

╔═════════════════════════════════════════════════════╗
║           COMPARATIVA FINAL · 3 PREGUNTAS          ║
╠══════════════════╦═════════╦═══════════╦═══════════╣
║  Métrica         ║  Básico ║ Pipeline  ║  Mejora   ║
╠══════════════════╬═════════╬═══════════╬═══════════╣
║  Hit Rate @3     ║   100%  ║    100%   ║  +0 pp      ║
║  MRR             ║   0.83  ║    1.00   ║  +16.7 pp     ║
║  Faithfulness    ║    N/A  ║    100%   ║   —         ║
╚══════════════════╩═════════╩═══════════╩═══════════╝

  Hit Rate: Fase 01 (Búsqueda Híbrida)  asegura que el doc esté en el top-3
  MRR:      Fase 02 (Reranking)         lo sube a la posición #1
  Faith.:   Fase 03 (CRAG)              verifica que lo que llega al LLM sea correcto


---
## 7 · Reto

Tienes todas las funciones del pipeline disponibles.
Tu objetivo es implementar `analizar_ganancia(preguntas)` que responda esta pregunta:

**¿Qué pregunta se benefició más del pipeline completo vs Dense solo, y en qué fase ocurrió la mayor mejora?**

La función debe:
1. Para cada pregunta, calcular el MRR con Dense solo, con solo Híbrida, y con el pipeline completo
2. Calcular la ganancia total de MRR por pregunta (`MRR_pipeline − MRR_dense`)
3. Identificar la pregunta con mayor ganancia
4. Determinar si esa ganancia vino principalmente de la Fase 01 (Híbrida) o de la Fase 02 (Reranking)

Funciones disponibles: `buscar_dense()`, `hibrida()`, `reranking()`, `pipeline_completo()`

**Output esperado** (los números variarán según el modelo, pero el formato debe ser este):
```
Query                                          Dense  Híbrida  Pipeline  Ganancia
-------------------------------------------------------------------------------------
¿Cómo derrotar al Lynel Escarlata?             0.33     1.00      1.00     +0.67  ← Fase 01 (Híbrida)
¿Cómo acceder al Templo del Agua de Zora?      0.33     1.00      1.00     +0.67  ← Fase 01 (Híbrida)
¿Cómo fabricar el Elixir Ígnifrugal?           0.50     1.00      1.00     +0.50  ← Fase 01 (Híbrida)

⊙ Mayor ganancia: '¿Cómo derrotar al Lynel Escarlata?' (+0.67 MRR)
```

In [ ]:
# ══ RETO ═════════════════════════════════════════════════════

def posicion_doc_correcto(resultados, id_correcto, k=5):
    """Devuelve la posición (1-based) del doc correcto en la lista de resultados.
    Si no aparece en los primeros k, devuelve k+1 (no encontrado)."""
    # Tu código aquí
    pass


def mrr_query(resultados, id_correcto):
    """Calcula el MRR para una sola query dado su lista de resultados."""
    # Tu código aquí
    pass


def analizar_ganancia(preguntas):
    """Para cada pregunta calcula MRR en tres puntos del pipeline y compara."""
    print(f"{'Query':<45}  {'Dense':>6}  {'Híbrida':>8}  {'Pipeline':>9}  {'Ganancia':>9}")
    print("-" * 85)

    mejor_pregunta = None
    mejor_ganancia = -1

    for p in preguntas:
        # Paso 1: Dense solo
        res_dense    = buscar_dense(p["query"], top_k=5)

        # Paso 2: Solo Híbrida (sin reranking)
        res_hibrida  = hibrida(p["query"], k=5)

        # Paso 3: Pipeline completo (Híbrida + Reranking)
        res_pipeline = pipeline_completo(p["query"], k=5)

        # Calcula los tres MRR
        mrr_d = mrr_query(res_dense,    p["correcto"])
        mrr_h = mrr_query(res_hibrida,  p["correcto"])
        mrr_p = mrr_query(res_pipeline, p["correcto"])

        ganancia = mrr_p - mrr_d

        # Determina en qué fase ocurrió la mayor mejora
        if (mrr_h - mrr_d) >= (mrr_p - mrr_h):
            fase_clave = "Fase 01 (Híbrida)"
        else:
            fase_clave = "Fase 02 (Reranking)"

        print(f"{p['query'][:44]:<45}  {mrr_d:>6.2f}  {mrr_h:>8.2f}  {mrr_p:>9.2f}  {ganancia:>+9.2f}  ← {fase_clave}")

        if ganancia > mejor_ganancia:
            mejor_ganancia  = ganancia
            mejor_pregunta  = p["query"]

    print(f"\n⊙ Mayor ganancia: '{mejor_pregunta}' (+{mejor_ganancia:.2f} MRR)")


analizar_ganancia(PREGUNTAS)

---
## 8 · Resumen y cierre

```
Pregunta del usuario
     │
     ▼  FASE 01 · Búsqueda Híbrida  →  KPI: Hit Rate
     Dense + BM25  →  RRF  →  Top-K candidatos
     │   (no pierde términos exactos ni lenguaje natural)
     │
     ▼  FASE 02 · Reranking  →  KPI: MRR
     Cross-Encoder  →  Top-3 reordenado
     │   (el doc correcto siempre en posición #1)
     │
     ▼  FASE 03 · CRAG  →  KPI: Faithfulness
     LLM Juez  →  Correcto / Ambiguo / Incorrecto
     │   (si falla, busca en la web antes de responder)
     │
     ▼  LLM generador con contexto verificado
     Respuesta sin alucinaciones
```

### Para llevar a producción
- **LLM Juez**: usa Claude Haiku o GPT-4o-mini (más baratos para evaluación)
- **Embeddings en español**: `intfloat/multilingual-e5-large` supera a MiniLM en español
- **Búsqueda web**: Tavily Search API se integra en una línea con LangChain
- **Evaluación real de Faithfulness**: frameworks como RAGAS o deepeval

### ⊙ Pregunta de cierre

> Piensa en un sistema RAG que conozcas o en uno que quisieras construir.
> ¿Cuál de las tres fases crees que daría el mayor impacto en ese caso concreto?
> ¿Por qué?